In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🌍 Multilingual RAG Bot

## What We're Building

A RAG system that can **retrieve documents in one language and answer in another**.
For example, retrieve from an English knowledge base but answer in Spanish, Arabic,
or any language — all using a single multilingual embedding model.

## The Challenge

```
Monolingual RAG:  Query in French → Embedding captures French → Misses English docs
Multilingual RAG: Query in French → Embedding captures MEANING → Finds English docs
```

## How Cross-Lingual Retrieval Works

```
┌──────────────────────────────────────────┐
│     Shared Multilingual Vector Space      │
│                                            │
│     "cat" (EN) ──●                        │
│     "gato" (ES)  ──●   ← Close together!  │
│     "قطة" (AR)   ──●                      │
│                                            │
│     "dog" (EN) ──●                        │
│     "perro" (ES) ──●   ← Also close!      │
└──────────────────────────────────────────┘
```

## Stack
- **Ollama embeddings** — `nomic-embed-text` has some multilingual capability
- **ChromaDB** — vector store
- **Ollama LLM** — generates answers in the requested language

## Step 1 — Imports & Multilingual Knowledge Base

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document

print("All imports successful!")

In [ ]:
# Knowledge base with documents in MULTIPLE languages
# In a real system, these could be manuals, articles, or wikis in different languages

multilingual_docs = [
    # English documents
    Document(
        page_content="The Great Wall of China is approximately 21,196 kilometers long. It was built over many centuries, starting from the 7th century BC. The most well-known sections were built during the Ming Dynasty (1368-1644).",
        metadata={"language": "en", "topic": "landmarks"}
    ),
    Document(
        page_content="Photosynthesis is the process by which green plants convert sunlight into chemical energy. Plants absorb carbon dioxide and water, using light energy to produce glucose and oxygen. This process is essential for life on Earth.",
        metadata={"language": "en", "topic": "science"}
    ),
    Document(
        page_content="The human heart beats approximately 100,000 times per day, pumping about 7,570 liters of blood. It has four chambers: two atria and two ventricles. The heart is roughly the size of a closed fist.",
        metadata={"language": "en", "topic": "biology"}
    ),

    # French documents
    Document(
        page_content="La Tour Eiffel mesure 330 mètres de hauteur. Elle a été construite par Gustave Eiffel pour l'Exposition universelle de 1889 à Paris. Initialement critiquée, elle est devenue le symbole le plus emblématique de la France.",
        metadata={"language": "fr", "topic": "landmarks"}
    ),
    Document(
        page_content="La cuisine française est inscrite au patrimoine culturel immatériel de l'UNESCO depuis 2010. Les plats emblématiques incluent le coq au vin, la ratatouille, et la bouillabaisse. La France produit plus de 400 variétés de fromage.",
        metadata={"language": "fr", "topic": "culture"}
    ),

    # Spanish documents
    Document(
        page_content="La selva amazónica es el bosque tropical más grande del mundo, cubriendo 5.5 millones de kilómetros cuadrados. Contiene el 10% de todas las especies conocidas en la Tierra. El río Amazonas es el más caudaloso del mundo.",
        metadata={"language": "es", "topic": "nature"}
    ),
    Document(
        page_content="El sistema solar tiene ocho planetas: Mercurio, Venus, Tierra, Marte, Júpiter, Saturno, Urano y Neptuno. Plutón fue reclasificado como planeta enano en 2006. Júpiter es el planeta más grande.",
        metadata={"language": "es", "topic": "science"}
    ),

    # German documents
    Document(
        page_content="Die deutsche Automobilindustrie ist weltführend. BMW, Mercedes-Benz, Volkswagen und Audi sind global bekannte Marken. Deutschland produziert jährlich etwa 4,7 Millionen Fahrzeuge und beschäftigt über 800.000 Menschen in der Branche.",
        metadata={"language": "de", "topic": "industry"}
    ),
    Document(
        page_content="Das Oktoberfest in München ist das größte Volksfest der Welt. Es findet jährlich von Mitte September bis Anfang Oktober statt. Über 6 Millionen Besucher kommen jedes Jahr und trinken etwa 7 Millionen Liter Bier.",
        metadata={"language": "de", "topic": "culture"}
    ),

    # Arabic documents
    Document(
        page_content="أهرامات الجيزة هي من عجائب الدنيا السبع القديمة. بُنيت منذ حوالي 4500 عام. الهرم الأكبر (هرم خوفو) يبلغ ارتفاعه 146 متراً وكان أطول بناء في العالم لمدة 3800 عام.",
        metadata={"language": "ar", "topic": "landmarks"}
    ),

    # Japanese documents
    Document(
        page_content="富士山は日本で最も高い山で、標高3,776メートルです。2013年にユネスコの世界文化遺産に登録されました。毎年約30万人が登山に訪れます。",
        metadata={"language": "ja", "topic": "landmarks"}
    ),
]

embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(multilingual_docs, embeddings, collection_name="multilingual")

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

langs = set(d.metadata['language'] for d in multilingual_docs)
print(f"Indexed {len(multilingual_docs)} documents in {len(langs)} languages: {sorted(langs)}")

## Step 2 — Cross-Lingual Retrieval Test

Query in one language? Can it find documents in other languages about the same topic?

In [ ]:
def cross_lingual_search(query: str, k: int = 4):
    """Test if queries in one language retrieve docs in other languages."""
    results = vectorstore.similarity_search_with_relevance_scores(query, k=k)

    print(f"\n🔍 Query: {query}")
    print(f"{'Lang':<6} {'Score':>6}  Content preview")
    print("-" * 65)
    for doc, score in results:
        lang = doc.metadata['language']
        preview = doc.page_content[:55].replace('\n', ' ')
        print(f"  {lang:<4} {score:>6.3f}  {preview}...")


# English query → should find docs about landmarks in ANY language
cross_lingual_search("famous monuments and landmarks in the world")

# French query → should find science docs in multiple languages
cross_lingual_search("comment les plantes produisent de l'énergie")

# Spanish query → should find nature/biology docs
cross_lingual_search("¿cuántas veces late el corazón por día?")

# German query → should find festival/culture docs
cross_lingual_search("traditionelle Feste und Kultur")

## Step 3 — Language Detection

Detect the query language so we can respond in the same language.

In [ ]:
detect_prompt = ChatPromptTemplate.from_template(
    """Detect the language of this text. Respond with ONLY the ISO 639-1
language code (e.g., en, fr, es, de, ar, ja, zh, ko, pt, ru, etc.).

Text: {text}

Language code:"""
)


def detect_language(text: str) -> str:
    chain = detect_prompt | llm | StrOutputParser()
    result = chain.invoke({"text": text}).strip().lower()
    # Extract 2-letter code
    for code in ["en", "fr", "es", "de", "ar", "ja", "zh", "ko", "pt", "ru", "it", "nl", "hi"]:
        if code in result:
            return code
    return "en"  # default


# Test language detection
test_texts = [
    "What is the tallest mountain in Japan?",
    "Quelle est la hauteur de la Tour Eiffel?",
    "¿Cuántos planetas hay en el sistema solar?",
    "Wie viele Autos produziert Deutschland?",
    "ما هي أهرامات الجيزة؟",
]

for text in test_texts:
    lang = detect_language(text)
    print(f"  [{lang}] {text[:50]}..." if len(text) > 50 else f"  [{lang}] {text}")

## Step 4 — Multilingual RAG Pipeline

1. Detect query language
2. Retrieve from all languages
3. Answer in the query's language

In [ ]:
LANGUAGE_NAMES = {
    "en": "English", "fr": "French", "es": "Spanish",
    "de": "German", "ar": "Arabic", "ja": "Japanese",
    "zh": "Chinese", "ko": "Korean", "pt": "Portuguese",
    "ru": "Russian", "it": "Italian", "hi": "Hindi",
}

multilingual_answer_prompt = ChatPromptTemplate.from_template(
    """You are a multilingual assistant. Answer the question using the
provided context. The context may be in various languages — translate
the relevant information as needed.

IMPORTANT: You MUST respond in {response_language}.

Context (may contain multiple languages):
{context}

Question: {question}

Answer in {response_language}:"""
)


def multilingual_rag(
    question: str,
    response_lang: str | None = None,
    k: int = 3
) -> str:
    """Full multilingual RAG pipeline.

    Args:
        question: Question in any language
        response_lang: Force response language (None = auto-detect from question)
        k: Number of documents to retrieve
    """
    # Step 1: Detect language
    query_lang = detect_language(question)
    target_lang = response_lang or query_lang
    lang_name = LANGUAGE_NAMES.get(target_lang, target_lang)

    print(f"\n{'='*60}")
    print(f"❓ {question}")
    print(f"🔤 Query language: {query_lang} | Response language: {target_lang} ({lang_name})")

    # Step 2: Cross-lingual retrieval
    docs = vectorstore.similarity_search(question, k=k)
    source_langs = [d.metadata['language'] for d in docs]
    print(f"📚 Retrieved from languages: {source_langs}")

    context = "\n\n".join(
        f"[{d.metadata['language'].upper()} | {d.metadata['topic']}] {d.page_content}"
        for d in docs
    )

    # Step 3: Generate answer in target language
    chain = multilingual_answer_prompt | llm | StrOutputParser()
    answer = chain.invoke({
        "context": context,
        "question": question,
        "response_language": lang_name
    })

    print(f"\n📝 Answer ({lang_name}):\n{answer}")
    return answer

In [ ]:
# English question — retrieves from multiple languages about landmarks
_ = multilingual_rag("What are the most famous landmarks in the world?")

In [ ]:
# French question — answers in French, retrieves from mixed languages
_ = multilingual_rag("Quelle est la hauteur de la Tour Eiffel et des pyramides?")

In [ ]:
# Spanish question — answers in Spanish
_ = multilingual_rag("¿Cuántos planetas hay en el sistema solar?")

In [ ]:
# English query but FORCE response in German
_ = multilingual_rag("Tell me about the German car industry", response_lang="de")

In [ ]:
# English query, response in Spanish — cross-lingual answer
_ = multilingual_rag("How does photosynthesis work?", response_lang="es")

## Step 5 — Language Coverage Analysis

In [ ]:
# Analyze how well cross-lingual retrieval works
test_queries = [
    ("famous buildings", "en"),
    ("monuments célèbres", "fr"),        # same meaning in French
    ("monumentos famosos", "es"),         # same meaning in Spanish
    ("berühmte Bauwerke", "de"),          # same meaning in German
]

print("Cross-lingual retrieval consistency test:")
print("Same concept queried in 4 languages — do they retrieve similar docs?\n")

for query, lang in test_queries:
    docs = vectorstore.similarity_search(query, k=3)
    source_langs = [f"{d.metadata['language']}:{d.metadata['topic']}" for d in docs]
    print(f"  [{lang}] \"{query}\" → {source_langs}")

## 🧠 Key Concepts Recap

| Concept | How |
|---------|-----|
| **Multilingual Embeddings** | Map text from any language into a shared vector space |
| **Cross-Lingual Retrieval** | Query in French, find relevant English docs |
| **Language Detection** | LLM identifies query language |
| **Response Language Control** | Instruct LLM to answer in specific language |

## Embedding Model Comparison for Multilingual

| Model | Languages | Dim | Free/Local |
|-------|-----------|-----|------------|
| `BGE-M3` | 100+ | 1024 | ✅ |
| `multilingual-e5-large` | 100+ | 1024 | ✅ |
| `nomic-embed-text` | ~10 | 768 | ✅ (Ollama) |
| OpenAI `text-embedding-3` | 50+ | 3072 | Paid API |
| Cohere `embed-multilingual` | 100+ | 1024 | Paid API |

## 🔧 Extensions

- **Query translation**: Translate query to corpus languages before retrieval
- **Language-specific collections**: Separate ChromaDB collections per language
- **Translation fallback**: If cross-lingual retrieval fails, translate and retry
- **Language-weighted scoring**: Boost same-language docs slightly